#### **04-engine**

The engine with nothing in front of it. `reset`, `step`, `done`, and a state
dict every bar.

**This is what you reach for when a rule cannot be written as arrays.** A
moving average is a function of the closes and can be computed before the run
starts. A trailing stop cannot: it is anchored to the price you were actually
filled at, and it moves with the highest price seen since then. Neither of
those exists until the run produces them.

The rule below enters on a crossover, which is precomputed because it can be,
and exits on a stop that ratchets up and never down, which is computed in the
loop because there is nowhere else it could be.

#### **Setup**

The environment is built before this cell runs. `docker compose up` installs
both libraries and starts a router. A pip install next to a router you started
yourself does the same thing.

So this cell checks and opens the client. If either library is missing, or
nothing answers on the router address, it stops here and prints the command to
run.

In [1]:
import os
import urllib.request

ROUTER_URL = os.environ.get(key="ROUTER_URL", default="http://127.0.0.1:8040")

try:
    import emsl
    import exchange_router_client
except ImportError as error:
    raise ImportError(f"{error.name} is missing; run 'docker compose up' from "
                      f"this repository, or pip install -r requirements.txt") from None


def router_is_up(url):
    try:
        with urllib.request.urlopen(url=f"{url}/status", timeout=2.0) as response:
            return response.status == 200
    except Exception:
        return False


if not router_is_up(url=ROUTER_URL):
    raise RuntimeError(f"no router answering on {ROUTER_URL}; run 'docker compose "
                       f"up' from this repository, or start one yourself and point "
                       f"ROUTER_URL at it")

client = exchange_router_client.ExchangeRouterClient(base_url=ROUTER_URL)

print("service    ", client.get_version(), "at", ROUTER_URL)
print("emsl       ", emsl.__version__)

service     2.5.6 at http://exchange-router-service:8040
emsl        1.3.1


#### **The candles**

One year of hourly bars, pinned so the run is over the same bars every time.
**`start` is the end of the window and the router walks backwards from it**, so
this asks for the year that ends on 1 January 2026, which is 2025.

A frame carries metadata, not just columns. `candles.attrs` holds what is
constant across the request: the venue, the symbol, the quote asset, the unit
volume is counted in, and the schema number. The unit matters more here than
anywhere else, because the engine compares an order size against a bar volume
to decide how much of it can fill. A feed shipping quote volume where the
engine expects base would inflate that cap by roughly the price, the fill limit
would stop binding, and nothing at all would say so.

In [2]:
import datetime

EXCHANGE = "binance"
MARKET   = "spot"
SYMBOL   = "BTCUSDT"
INTERVAL = "1h"

ANCHOR = int(datetime.datetime(year=2026, month=1, day=1,
                               tzinfo=datetime.timezone.utc).timestamp() * 1000)
BARS   = 8760

candles = client.get_candles(
    exchange=EXCHANGE,
    market_type=MARKET,
    symbol=SYMBOL,
    interval=INTERVAL,
    limit=BARS,
    start=ANCHOR,
)

print(f"{len(candles)} bars, {candles.index[0]} to {candles.index[-1]}")
print(f"columns  {', '.join(candles.columns)}")

for key in ("exchange", "market_type", "symbol", "quote", "volume_unit",
            "schema_version"):
    print(f"  {key:<16} {candles.attrs[key]}")

8760 bars, 2025-01-01 01:00:00+00:00 to 2026-01-01 00:00:00+00:00
columns  open, high, low, close, volume, volume_usd
  exchange         binance
  market_type      spot
  symbol           BTCUSDT
  quote            USDT
  volume_unit      base
  schema_version   3


#### **The loop**

Three calls. `reset` returns the first state, `step` advances one bar and
returns the next, `done` says when the series ran out. There is no framework
between you and that.

Everything the account knows is in the state dict, and two fields are what
make a path-dependent rule possible at all. `avg_entry` is the weighted price
you were actually filled at, and `position` is the size you are holding. Neither
can be computed in advance, because both are answers to what the run did.

`report=True` is not the default. The engine keeps a trade log only when
asked, because keeping one costs something on every fill and the reinforcement
learning path runs thousands of environments that never read it.

In [3]:
VENUE = emsl.Market(
    kind="spot",
    quote=10_000.0,
    fee_taker=0.0006,
    fee_maker=0.0002,
    slippage_bps=2.0,
)

peek  = VENUE.engine(candles=candles, report=True)
state = peek.reset()

print(f"{len(state)} fields in state, at bar {state['tick_index']}\n")
for key in sorted(state):
    print(f"  {key:<16} {state[key]}")

16 fields in state, at bar 0

  avg_entry        0.0
  bar_close        93607.74
  bar_high         94408.72
  bar_low          93578.77
  bar_open         94401.13
  bar_volume       586.53456
  base             0.0
  equity           10000.0
  funding_paid     0.0
  mark_price       93607.74
  open_orders      []
  position         0.0
  quote            10000.0
  realized_pnl     0.0
  tick_index       0
  unrealized_pnl   0.0


#### **The stop that has to be in the loop**

**A trailing stop is one order that moves, not many orders placed.** Nothing
links resting orders into a group, so calling `stop()` every bar rests a new one
every bar. The one that eventually fills leaves its siblings alive, and on a
perpetual those go on to open a position in the other direction.

`replace` cancels and re-rests as one operation, so it cannot leave two alive,
and it **returns `None` once the order has already filled**. That is how the
loop finds out the stop is gone without having to ask.

`reduce_only=True` means even a leaked stop can only shrink the position rather
than open one.

The trigger only ever moves up. `wanted > armed` is the line that makes
this a trailing stop instead of one that follows the price down as well as
up.

The entries are still precomputed. `ema` and `crossover` are causal functions
of the closes, so computing them once before the loop is faster and changes
nothing. **What can be an array is an array, and what cannot is in the
loop.**

In [4]:
FAST     = 20
SLOW     = 60
WEIGHT   = 0.95
MAX_LOSS = 0.05    #  the stop never sits worse than this under the entry
TRAIL    = 0.03    #  once armed, follow this far under the high water mark
ARM_AT   = 0.02    #  start trailing only once the trade is this far in front

engine = VENUE.engine(candles=candles, report=True)

fast = emsl.ta.ema(values=engine.closes, length=FAST)
slow = emsl.ta.ema(values=engine.closes, length=SLOW)
up   = emsl.ta.crossover(values=fast, other=slow)

log = emsl.plot.Recorder(bars=engine)

state   = engine.reset()
stop_id = None
armed   = None
peak    = None

while not engine.done():
    i = state["tick_index"]

    if state["position"] == 0.0:
        stop_id = armed = peak = None

        if i >= SLOW and up[i]:
            engine.market_buy(size=engine.qty_from_weight(fraction=WEIGHT))

    else:
        entry = state["avg_entry"]
        peak  = state["bar_high"] if peak is None else max(peak, state["bar_high"])

        floor  = entry * (1.0 - MAX_LOSS)
        trail  = peak * (1.0 - TRAIL)
        wanted = max(floor, trail) if peak >= entry * (1.0 + ARM_AT) else floor

        if stop_id is None:
            stop_id = engine.stop(side="sell", size=state["position"],
                                  trigger=wanted, reduce_only=True)
        elif wanted > armed:
            stop_id = engine.replace(order_id=stop_id, trigger=wanted)

        armed = wanted

    holding = state["position"] > 0.0
    log.at_bar(state,
               stop=armed if holding else float("nan"),
               holding=holding)

    state = engine.step()

print(f"{engine.num_fills()} fills, bust={engine.is_bust()}")

93 fills, bust=False


#### **What the engine kept**

The raw engine hands its pieces back separately rather than as one result
object: a trade log, an equity curve, and the same stats a backtest reports.

**The equity curve is one shorter than the frame, and that is a contract rather
than an off-by-one.** The engine records a point only on a real advance, so
entry `i` is the account at tick `i + 1`. A chart accepts both lengths and draws
the short one from bar 1, which is why the equity line below lands where it
should without anything being padded.

**`Recorder` declares alignment where you write the value**, which is where the
meaning is obvious rather than three cells later. `at_bar` says entry `i` is bar
`i`; `at_next` says it is bar `i + 1`. Getting it wrong draws an exposure story
one bar to the right of the candles that explain it, and nothing raises.

A bar the loop skipped is a gap, and a boolean one is `False` rather than NaN,
which matters because `bool(nan)` is `True` and a mask padded like a float
reports the event it was looking for. It costs one array write per value per
bar, so leave it out of a strategy you mean to sweep across hundreds of trials.

In [5]:
trades = engine.trades()
curve  = engine.equity_curve()
stats  = engine.stats()

print(f"{len(trades)} trades, equity curve {len(curve)} points "
      f"for {len(candles)} bars\n")

for key in ("total_return_pct", "sharpe", "max_drawdown_pct",
            "num_trades", "exposure_pct"):
    print(f"  {key:<20} {stats[key]:>12,.2f}")

print(f"\nrecorded {log.keys()}, one value per bar")
print(f"  stop     {log['stop'].dtype}, {len(log['stop'])} values")
print(f"  holding  {log['holding'].dtype}, {len(log['holding'])} values")

46 trades, equity curve 8759 points for 8760 bars

  total_return_pct           -31.37
  sharpe                      -0.26
  max_drawdown_pct            32.78
  num_trades                  46.00
  exposure_pct                56.72

recorded ['stop', 'holding'], one value per bar
  stop     float64, 8760 values
  holding  bool, 8760 values


#### **Looking at it**

There is no `run=` here, because there is no `BacktestResult` to pass. So
the chart draws only what it is given: no fill arrows, no trade table. The
shading is doing the job those would have done, and it comes from the boolean
the loop recorded rather than from anything the chart worked out.
Naming it puts it in the legend, so the crosshair says whether the bar
under it was one the loop was holding.

The stop line is the one to look at. It exists only while a position does,
and a NaN is a gap rather than a dropped row, so the line stops at each exit
instead of drawing a straight segment across the flat stretch to the next entry.
Dropping those rows would join the neighbours and paint a stop that was never
there.

Watch it ratchet: flat under the entry until the trade is far enough in front,
then stepping up with the high water mark and never back down.

Over a full year of hourly candles the stop would be a hairline, which is
what `focus=1200` is for.

In [6]:
emsl.chart(
    frame=candles,
    marks=[
        emsl.plot.Line(values=fast, name=f"EMA {FAST}"),
        emsl.plot.Line(values=slow, name=f"EMA {SLOW}", style="dashed"),
        emsl.plot.Line(values=log["stop"], name="trailing stop", width=1),
        emsl.plot.Background(values=log["holding"], name="holding",
                             fill=["#2fe0a826", "#2fe0a803"]),
        emsl.plot.Line(values=curve, name="equity", panel="equity"),
    ],
    panels=[
        emsl.plot.Panel(name="volume", show=False),
        emsl.plot.Panel(name="equity", weight=2.0),
    ],
    focus=1200,
    title="one order moved, not many placed",
    notes=[
        ["setting", "value", "what it does"],
        ["FAST", FAST, "fast EMA length"],
        ["SLOW", SLOW, "slow EMA length"],
        ["WEIGHT", WEIGHT, "fraction of equity an entry takes"],
        ["MAX_LOSS", MAX_LOSS, "the stop never sits worse than this under entry"],
        ["TRAIL", TRAIL, "once armed, follow this far under the high water mark"],
        ["ARM_AT", ARM_AT, "start trailing once the trade is this far in front"],
    ],
).show()

#### **Closing**

**Reach for the loop when the rule reads the run.** A stop anchored to the fill
price, bars since entry, a cooldown after a loss, a size that scales with
current equity, an accumulation that resets when the position goes flat. None of
those are functions of the price series, so no amount of vectorising computes
them ahead of time.

**Do not reach for it to avoid lookahead.** Slicing `engine.closes[:i + 1]` is
what prevents that, and it works in a strategy class just as well. The views are
the whole series in the loop too, so `engine.closes[i + 50]` is readable here as
well and nothing stops you.

Two things the loop makes you responsible for. `close()` is an ordinary
market order, so on a thin bar the volume cap can fill part of it and cancel the
rest, leaving a residual position: check `state["position"]` rather than
assuming. And a bar that reaches both a stop and a target books the stop,
because the engine cannot see the path inside a bar and pessimism is the honest
direction for an unobservable ordering.

What is not here. `limit_buy` and `limit_sell` rest as makers and pay the
maker fee rather than the taker one. `order` is the primitive the shortcuts
wrap, and the only call that sets `post_only` and the time in force. And
`cancel_all` exists for the moment a rule decides to stand down entirely.

`client.close()` shuts down the background thread the client runs its async loop
on, and its connection pool.

In [7]:
client.close()